In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split , GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_squared_error, r2_score

In [3]:
df= pd.read_csv('regression_data.csv',header=None)

FileNotFoundError: [Errno 2] No such file or directory: 'regression_data.csv'

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.describe()

In [ ]:

df.sample(5)


In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
df.isna().any().sum()

In [ ]:
for col in df:
    if df[col].isna().sum() > 0 :
         print(f"{col}: {df[col].isna().sum()} ")



In [ ]:
 df.dropna(axis=1,inplace=True)
df

In [ ]:
df.isnull().sum()

In [ ]:
df.isna().sum().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
# Select numerical columns only
numeric_features = df.select_dtypes(include=[np.int64, np.float64]).columns.tolist()
numeric_features.remove(80)  # Don't check outliers in the target column
numeric_features

In [ ]:
# Function to calculate outliers using IQR
def detect_outliers_iqr(data, feature):
    Q1 = data[feature].quantile(0.25)
    Q3 = data[feature].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = data[(data[feature] < lower_bound) | (data[feature] > upper_bound)]
    return outliers

# Check outliers in each numerical column
for col in numeric_features:
    outliers = detect_outliers_iqr(df, col)
    print(f"{col}: {len(outliers)} outliers")

In [ ]:
def cap_outliers_iqr(data, feature):
    Q1 = data[feature].quantile(0.25)
    Q3 = data[feature].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    data[feature] = np.where(data[feature] < lower_bound, lower_bound,
                     np.where(data[feature] > upper_bound, upper_bound, data[feature]))
    return data

# Apply capping to all numeric features (except target)
for col in numeric_features:
    df = cap_outliers_iqr(df, col)

In [ ]:
# Check outliers in each numerical column
for col in numeric_features:
    outliers = detect_outliers_iqr(df, col)
    print(f"{col}: {len(outliers)} outliers")

In [ ]:
categorical_features = df.select_dtypes(include=[object]).columns.tolist()
for col in categorical_features:
    print(f"{col}: {df[col].nunique()} unique values")
    print(df[col].value_counts())
    print("-----")

In [ ]:
from sklearn.preprocessing import LabelEncoder
list=[2,5,7,8,9,10,11,12,13,14,15,16,21,22,23,24,27,28,29,39,40,41,53,55,65,78,79]
le = LabelEncoder()
for element in list:
    df[element] = le.fit_transform(df[element])


df.head()

In [ ]:
correlations = df.corr()[80].drop(80)

In [ ]:
highest_corr_feature = correlations.abs().idxmax()
highest_corr_value = correlations[highest_corr_feature]

print(f"the highest correlation is column number: {highest_corr_feature} and correlation coef is {highest_corr_value:.4f}")


In [ ]:
correlations.abs().sort_values(ascending=False).plot(kind='bar', figsize=(10,6))
plt.title('Feature Correlation with Target')
plt.ylabel('Absolute Correlation Coefficient')
plt.xlabel('Features')
plt.show()

In [ ]:
# Define the feature (independent variable) and target (dependent variable)
x = df[[17]]
y = df[80]
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [ ]:
x_train

In [ ]:
# Create and train the model
model = LinearRegression()
model.fit(x_train, y_train)

In [ ]:
# Predictions
y_train_pred = model.predict(x_train)
y_test_pred = model.predict(x_test)

In [ ]:
print('train Accuracy : ', model.score(x_train, y_train))
print('test Accuracy : ', model.score(x_test, y_test))

In [ ]:
x = df.drop(columns=[80])
y = df[80]
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [ ]:
# Create and train the model
model = LinearRegression()
model.fit(x_train, y_train)

In [ ]:
# Predictions
y_train_pred = model.predict(x_train)
y_test_pred = model.predict(x_test)

In [ ]:
print('train Accuracy : ', model.score(x_train, y_train))
print('test Accuracy : ', model.score(x_test, y_test))

In [ ]:
# Fit a polynomial regression model with a high degree (overfitting scenario)
poly_features = PolynomialFeatures(degree=2)  # High-degree polynomial
X_train_poly = poly_features.fit_transform(x_train)
X_test_poly = poly_features.transform(x_test)

model = LinearRegression()
model.fit(X_train_poly, y_train)

In [ ]:
# Predictions
y_train_pred = model.predict(X_train_poly)
y_test_pred = model.predict(X_test_poly)

In [ ]:
# Calculate R² for training and test sets
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

In [ ]:
# Print R²
print(f"Training R² (Overfitting): {train_r2:.2f}")
print(f"Test R² (Overfitting): {test_r2:.2f}")

In [ ]:
from sklearn.linear_model import Lasso, Ridge

In [ ]:
# Apply L1 (Lasso) Regularization
lasso_model = Lasso(alpha=0.1)  # The alpha value controls the amount of regularization
lasso_model.fit(X_train_poly, y_train)

# Predictions using Lasso
y_train_pred_lasso = lasso_model.predict(X_train_poly)
y_test_pred_lasso = lasso_model.predict(X_test_poly)

# Calculate R² score for Lasso
train_r2_lasso = r2_score(y_train, y_train_pred_lasso)
test_r2_lasso = r2_score(y_test, y_test_pred_lasso)

print(f"Lasso (L1) Training R²: {train_r2_lasso:.2f}")
print(f"Lasso (L1) Test R²: {test_r2_lasso:.2f}")

In [ ]:
# Apply L2 (Ridge) Regularization
ridge_model = Ridge(alpha=0.1)  # The alpha value controls the amount of regularization
ridge_model.fit(X_train_poly, y_train)

# Predictions using Ridge
y_train_pred_ridge = ridge_model.predict(X_train_poly)
y_test_pred_ridge = ridge_model.predict(X_test_poly)

# Calculate R² score for Ridge
train_r2_ridge = r2_score(y_train, y_train_pred_ridge)
test_r2_ridge = r2_score(y_test, y_test_pred_ridge)

print(f"Ridge (L2) Training R²: {train_r2_ridge:.2f}")
print(f"Ridge (L2) Test R²: {test_r2_ridge:.2f}")